# Análisis de la esperanza de vida

Dos conjuntos de datos que exploran la esperanza de vida global:
- **Gapminder** (1952-2007): country, year, population, continent, lifeExp, gdpPercap
- **WHO Life Expectancy** (2000-2015): 193 países, 22 indicadores (mortalidad, IMC, PIB, escolaridad, etc.)

Este cuaderno de trabajo demuestra la importación y el análisis de archivos CSV tanto en **Python** como en **R**.

## 1. Configuración: instalar paquetes y descargar conjuntos de datos

In [ ]:
import micropip
await micropip.install(['pandas', 'plotly'])
print('pandas + plotly instalados')

import pyodide.http, os

datasets = {
    "gapminder.csv": "https://raw.githubusercontent.com/resbaz/r-novice-gapminder-files/master/data/gapminder-FiveYearData.csv",
    "who_life_expectancy.csv": "https://raw.githubusercontent.com/Sid-149/Life-Expectancy-Predictor-Comparative-Analysis/main/Notebooks/Life%20Expectancy%20Data.csv"
}

os.makedirs("/shared/data", exist_ok=True)

for name, url in datasets.items():
    path = f"/shared/data/{name}"
    if os.path.exists(path):
        print(f"Ya existe: {path}")
    else:
        resp = await pyodide.http.pyfetch(url)
        text = await resp.string()
        with open(path, "w") as f:
            f.write(text)
        lines = text.count("\n")
        print(f"Descargado {name}: {lines} líneas")

## 2. Gapminder: exploración con Python

In [ ]:
import pandas as pd

gap = pd.read_csv("/shared/data/gapminder.csv")
print(f"Dimensiones: {gap.shape}")
print(f"Continentes: {sorted(gap['continent'].unique())}")
print(f"Rango de años: {gap['year'].min()}-{gap['year'].max()}")
print()
gap.describe()

In [ ]:
import plotly.express as px
import json, js
from plotly.utils import PlotlyJSONEncoder

def plain_plotly(value):
    if hasattr(value, 'tolist'):
        return value.tolist()
    if isinstance(value, dict):
        return {key: plain_plotly(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [plain_plotly(item) for item in value]
    return value

def show_plotly(fig):
    payload = plain_plotly(fig.to_plotly_json())
    js.renderPlot(json.dumps({"traces": payload["data"], "layout": payload["layout"]}, cls=PlotlyJSONEncoder))

# Esperanza de vida a lo largo del tiempo por continente
avg = gap.groupby(['year', 'continent'])['lifeExp'].mean().reset_index()
fig = px.line(avg, x='year', y='lifeExp', color='continent',
              title='Esperanza de vida por continente (1952-2007)',
              labels={'lifeExp': 'Esperanza de vida (años)', 'year': 'Año'})
fig.update_layout(template='plotly_dark')
show_plotly(fig)

In [ ]:
# PIB vs. esperanza de vida (2007), tamaño de burbuja = población
g2007 = gap[gap['year'] == 2007]
fig = px.scatter(g2007, x='gdpPercap', y='lifeExp', size='pop',
                 color='continent', hover_name='country',
                 log_x=True, size_max=50,
                 title='PIB vs. esperanza de vida (2007)',
                 labels={'gdpPercap': 'PIB per cápita (log)', 'lifeExp': 'Esperanza de vida'})
fig.update_layout(template='plotly_dark')
show_plotly(fig)

## 3. Gapminder: exploración con R

In [ ]:
gap <- read.csv("/shared/data/gapminder.csv")
str(gap)
summary(gap$lifeExp)

In [ ]:
# Distribución de la esperanza de vida por continente (diagrama de caja)
par(bg = "#1e1e1e", fg = "white", col.axis = "white",
    col.lab = "white", col.main = "white")
boxplot(lifeExp ~ continent, data = gap,
        main = "Esperanza de vida por continente",
        xlab = "Continente", ylab = "Esperanza de vida (años)",
        col = c("#636EFA", "#EF553B", "#00CC96", "#AB63FA", "#FFA15A"),
        border = "white")

In [ ]:
# Los 10 países con mayor mejora en la esperanza de vida (1952 vs. 2007)
early <- gap[gap$year == 1952, c("country", "lifeExp")]
late  <- gap[gap$year == 2007, c("country", "lifeExp")]
merged <- merge(early, late, by = "country", suffixes = c("_1952", "_2007"))
merged$improvement <- merged$lifeExp_2007 - merged$lifeExp_1952
top10 <- head(merged[order(-merged$improvement), ], 10)

par(bg = "#1e1e1e", fg = "white", col.axis = "white",
    col.lab = "white", col.main = "white", mar = c(5, 10, 4, 2))
barplot(top10$improvement, names.arg = top10$country,
        horiz = TRUE, las = 1,
        main = "Top 10: incremento en la esperanza de vida (1952-2007)",
        xlab = "Años ganados",
        col = "#00CC96", border = NA)

## 4. WHO Life Expectancy: exploración con Python

In [ ]:
who = pd.read_csv("/shared/data/who_life_expectancy.csv")
print(f"Dimensiones: {who.shape}")
print(f"Columnas: {list(who.columns)}")
print(f"\nValores faltantes (primeros 5):")
print(who.isnull().sum().sort_values(ascending=False).head())
print()
who.head()

In [ ]:
# En desarrollo vs. desarrollados: distribuciones de esperanza de vida previamente agrupadas
# Las coordenadas de barra explícitas se representan de forma coherente mediante el puente de Plotly en el navegador.
import numpy as np
life = who.dropna(subset=['Life expectancy'])
edges = np.linspace(life['Life expectancy'].min(), life['Life expectancy'].max(), 41)
bin_width = edges[1] - edges[0]
hist_rows = []
for status, group in life.groupby('Status'):
    counts, _ = np.histogram(group['Life expectancy'], bins=edges)
    hist_rows.extend({
        'Life expectancy': float(left + bin_width / 2),
        'Count': int(count),
        'Status': status
    } for left, count in zip(edges[:-1], counts))

hist = pd.DataFrame(hist_rows)
fig = px.bar(hist, x='Life expectancy', y='Count', color='Status',
             barmode='overlay', opacity=0.7,
             title='Esperanza de vida: en desarrollo vs. desarrollados',
             labels={'Life expectancy': 'Esperanza de vida (años)'})
fig.update_traces(width=float(bin_width * 0.92))
fig.update_layout(template='plotly_dark', bargap=0.03)
show_plotly(fig)

In [ ]:
# Escolaridad vs. esperanza de vida
w2014 = who[who['Year'] == 2014].dropna(subset=['Schooling', 'Life expectancy'])
fig = px.scatter(w2014, x='Schooling', y='Life expectancy',
                 color='Status', hover_name='Country',
                 title='Escolaridad vs. esperanza de vida (2014)',
                 labels={'Life expectancy': 'Esperanza de vida (años)',
                         'Schooling': 'Años de escolaridad'})
fig.update_layout(template='plotly_dark')
show_plotly(fig)

## 5. WHO Life Expectancy: exploración con R

In [ ]:
who <- read.csv("/shared/data/who_life_expectancy.csv")
str(who)
cat("\nPaíses:", length(unique(who$Country)))
cat("\nRango de años:", range(who$Year))

In [ ]:
# Correlación: mortalidad adulta vs. esperanza de vida
par(bg = "#1e1e1e", fg = "white", col.axis = "white",
    col.lab = "white", col.main = "white")
plot(who$Adult.Mortality, who$Life.expectancy,
     pch = 16, cex = 0.5,
     col = ifelse(who$Status == "Developed", "#636EFA80", "#EF553B80"),
     main = "Mortalidad adulta vs. esperanza de vida",
     xlab = "Mortalidad adulta (por cada 1000)",
     ylab = "Esperanza de vida (años)")
legend("topright", legend = c("Desarrollado", "En desarrollo"),
       col = c("#636EFA", "#EF553B"), pch = 16, text.col = "white")

In [ ]:
# Modelo lineal simple: ¿qué predice la esperanza de vida?
who_clean <- na.omit(who[, c("Life.expectancy", "Schooling",
                              "Adult.Mortality", "GDP", "BMI")])
model <- lm(Life.expectancy ~ Schooling + Adult.Mortality + log1p(GDP) + BMI,
            data = who_clean)
summary(model)

## Conclusiones principales

- La esperanza de vida ha aumentado a nivel mundial, pero persisten grandes disparidades entre continentes
- El PIB y la escolaridad son predictores positivos fuertes de la esperanza de vida
- La mortalidad adulta es el predictor negativo más fuerte
- Los países en desarrollo muestran una varianza mucho más amplia en los resultados